# 21 LaCAM / LaCAM*：懒惰构型搜索


本课按百科条目写。LaCAM 在联合构型图上搜，但后继不枚举 \(5^k\)。调度台在最后。

---

## 1. 定义

**LaCAM**（Lazy Constraints Addition search for MAPF，Okumura, AAAI 2023）：在 **联合构型图** 上搜，后继用 PIBT（等）作 **configuration generator**，需要时再补后继（lazy）。

**LaCAM\***：anytime，继续搜，代价不增，累加转移代价下可趋向最优。

- 输入：图、起终点、扩展 cap。
- 输出：构型序列拆成的各车路径。

---

## 2. 和联合 A* / PIBT

| | 联合 A\* | PIBT | LaCAM |
|--|--|--|--|
| 后继 | 全枚举 \(5^k\) | 无搜索，逐步贪心 | PIBT 给一个 + 少量补边 |
| 死锁 | 能搜出来（若可解） | 可能卡 | 搜其它构型逃出 |
| 最优 | 是 | 否 | LaCAM 否；LaCAM\* anytime |

---

## 3. 不变量

- 栈上是联合位置元组。
- `came` 防重复构型。
- `_config_ok`：同一格不能两车。
- LaCAM\* 的 `history` 非增。

---

## 4. 伪代码（教学版）

```
stack ← [start_config]
came[start] ← nil
while stack:
    Q ← pop
    if Q = goal: 回溯
    Q' ← PIBT 一步(Q)          # 生成器
    if Q' 未见过: 入栈
    可选：枚举单车一步补完备性
```

LaCAM*：找到解后继续，接受更好的 PIBT 全轨迹。

---

## 5. 手算

两车已在目标：栈弹出即停，expanded=1。

PIBT 一步就会死锁的构型：生成器返回原地，came 挡住，单车挪一步的补边可能把其中一车让到旁边，搜索继续。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：构型栈 + PIBT 一步


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.algos.rules import pibt
from lib.agents import soc


def lacam_tiny(graph, starts, goals, cap=200, verbose=True):
    agents = list(starts)
    start = tuple(starts[a] for a in agents)
    goal = tuple(goals[a] for a in agents)
    stack = [start]
    came = {start: None}
    expanded = 0
    while stack and expanded < cap:
        cur = stack.pop()
        expanded += 1
        if verbose and expanded <= 8:
            print(f"弹出构型#{expanded}", cur, "栈长", len(stack))
        if cur == goal:
            chain = []
            x = cur
            while x is not None:
                chain.append(x)
                x = came[x]
            chain.reverse()
            paths = {a: [st[i] for st in chain] for i, a in enumerate(agents)}
            return {"found": True, "paths": paths, "soc": soc(paths), "expanded": expanded}
        dummy = {a: cur[i] for i, a in enumerate(agents)}
        step = pibt(graph, dummy, goals, steps=1)
        nxt = tuple(step["paths"][a][-1] for a in agents)
        if nxt not in came:
            came[nxt] = cur
            stack.append(nxt)
        if expanded < cap // 2:
            for i, a in enumerate(agents):
                for n, _ in list(graph.neighbors(cur[i])) + [(cur[i], 0)]:
                    cfg = list(cur)
                    cfg[i] = n
                    tcfg = tuple(cfg)
                    if tcfg not in came and len(set(tcfg)) == len(tcfg):
                        came[tcfg] = cur
                        stack.append(tcfg)
    return {"found": False, "paths": {}, "expanded": expanded}


g, starts, goals = two_agent_tunnel()
r = lacam_tiny(g, starts, goals)
print("tiny found", r["found"], "SOC", r.get("soc"), "expanded", r["expanded"])


In [ ]:
from lib.algos.rules import lacam, lacam_star

lib_r = lacam(g, starts, goals)
print("LaCAM", lib_r["found"], lib_r["soc"], lib_r["expanded"])
star = lacam_star(g, starts, goals)
print("LaCAM* 历史", star["history"])
print("非增", all(star["history"][i] <= star["history"][i - 1] + 1e-9 for i in range(1, len(star["history"]))))


## 7. 逐行

| 块 | 作用 |
|----|------|
| 构型元组 | 联合位置 |
| `pibt(..., steps=1)` | 一个后继 |
| 单车挪一步枚举 | 小图完备 |
| `lacam_star` 多轮 PIBT | anytime 非增 |

## 常见 bug

1. 生成器总返回同一构型，搜索原地打转（came 可挡，但要有补边否则停住）。
2. 把 LaCAM 当 PIBT 逐步执行器，忘了高层栈。
3. 构型用 list 当 dict 键——必须 tuple。

## 条目小结

| 项目 | 内容 |
|------|------|
| 图 | 联合构型 |
| 后继 | lazy（PIBT） |
| LaCAM\* | anytime |
| 下一课 | 谁去哪个目标：指派 |

## 练习

1. 为何 PIBT 生成器「一个后继」不够完备？补边在补什么？
2. LaCAM\* 历史非增，最优性要 n→∞ 还是 cap→∞？
3. 3 车时联合 A\* cap 400 失败，LaCAM 为什么还能走？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), lib_r, "LaCAM")
studio
